[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_docking_screen.ipynb)

# Dock a compound library into CpABC1

**Blue group · Cryptosporidiosis**

Docking places a molecule inside a protein's binding site and scores how well it fits. In this notebook you upload the CpABC1-silymarin complex, check that the docking software can put silymarin back where it was, and then dock a library of silymarin analogues into the same pocket to see whether any of them score better.

## What you will do

- Upload the CpABC1-silymarin complex and split it into receptor and ligand
- Redock silymarin and check that the pose comes back where it started
- Build 3D structures for a library of silymarin analogues
- Dock the whole library into the same pocket
- Rank the compounds and see which ones beat silymarin

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. Don't change it.

In [ ]:
PROJECT = "blue"
NEEDS_GPU = True
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose T4 GPU, and run this cell again.")

## 1. Get the docking software

We use **gnina**, a docking program that searches for the positions a molecule can take inside a
pocket. For each position (a *pose*) it reports the classical Vina affinity and two scores from a
convolutional neural network, which is what makes gnina different from older docking programs.

gnina is not a Python package: it is a single Linux program, so we download it and make it
executable rather than installing it. It needs a GPU, which is why this notebook asks for the T4
runtime. The file is 1.4 GB and takes a couple of minutes to arrive.

The program was built against version 12 of CUDA, the toolkit that lets programs use the GPU, and
Colab has since moved to version 13. Most of the version 12 libraries are still there, but one is
not, so we fetch it into a folder of its own and tell the program where to look. Nothing else in
the runtime is touched.

> **Note:** if this cell ends with `error while loading shared libraries: libsomething.so.12`,
> Colab has dropped another of those libraries. Add its package (`nvidia-<something>-cu12`) to the
> install line below, and tell the workshop organisers.

In [ ]:
import glob, os, stat, subprocess, sys, urllib.request

WORK = "work"  # everything this notebook produces goes here
os.makedirs(WORK, exist_ok=True)
GNINA = f"{WORK}/gnina"

if not os.path.exists(GNINA):
    urllib.request.urlretrieve("https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2", GNINA)
    os.chmod(GNINA, os.stat(GNINA).st_mode | stat.S_IEXEC)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", f"{WORK}/cuda12",
                    "nvidia-nvtx-cu12==12.1.105"], check=True)

libs = glob.glob("/usr/local/lib/python3.*/dist-packages/nvidia/*/lib/*.so*") + glob.glob(f"{WORK}/cuda12/nvidia/*/lib/*.so*")
ENV = {**os.environ, "LD_LIBRARY_PATH": ":".join(sorted({os.path.abspath(os.path.dirname(p)) for p in libs}))}
print(subprocess.run([GNINA, "--version"], capture_output=True, text=True, env=ENV).stdout.strip())

## 2. Upload the complex

Docking needs two things: the protein (the *receptor*) and a molecule already sitting in the pocket
(the *reference ligand*), which tells the program where to search. Both are in the same file here:
the CpABC1-silymarin complex the group prepared, `CpABC1-Silymarin.pdb` in the Drive folder
`Projects/BlueTeam/Data`.

Run the cell, click **Choose Files** and pick that file. It is about 2 MB, so the upload takes a few
seconds. Any PDB file that holds a protein with one bound molecule works here, so you can come back
later and try a different complex.

> **Note:** the file is uploaded to this Colab session only. It disappears when the session ends,
> and it is never added to the workshop repository.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    complex_path = f"{WORK}/{name}"
    with open(complex_path, "wb") as f:
        f.write(uploaded[name])
else:
    complex_path = f"{WORK}/CpABC1-Silymarin.pdb"  # outside Colab, put the file here yourself
print(f"{complex_path} | {os.path.getsize(complex_path) / 1e6:.1f} MB")

## 3. Split the complex into receptor and ligand

A PDB file uses `ATOM` lines for the protein and `HETATM` lines for everything else: the bound
molecule, but also water and ions. We keep the protein as the receptor, and take the largest group
of `HETATM` lines belonging to one residue as the ligand. Water is skipped.

In [ ]:
lines = open(complex_path).read().splitlines()
protein = [line for line in lines if line.startswith("ATOM")]

residues = {}
for line in lines:
    if line.startswith("HETATM") and line[17:20].strip() not in ("HOH", "WAT"):
        residues.setdefault(line[17:26], []).append(line)  # residue name, chain and number

residue, ligand_lines = max(residues.items(), key=lambda item: len(item[1]))
with open(f"{WORK}/receptor.pdb", "w") as f:
    f.write("\n".join(protein) + "\nEND\n")
print(f"receptor: {len(protein)} atoms | ligand ({residue.strip()}): {len(ligand_lines)} atoms")

A PDB file records where the atoms are but not how they are bonded, so RDKit works the bonds out
from the distances between atoms. Printing the ligand as a SMILES string, the text form of a
molecule, is the quickest way to confirm that we really pulled out silymarin and not an ion.

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds

ligand = Chem.MolFromPDBBlock("\n".join(ligand_lines), removeHs=False)
rdDetermineBonds.DetermineBondOrders(ligand, charge=0)
Chem.MolToMolFile(ligand, f"{WORK}/ligand.sdf")
print(Chem.MolToSmiles(Chem.RemoveHs(ligand)))

## 4. Redock silymarin

Before trusting any score, check that the setup works: take silymarin out of the pocket, let gnina
search for its position again, and see whether it finds its way back. This is called *redocking*.

`--autobox_ligand` tells gnina to search in a box drawn around the reference ligand, so it looks at
the right pocket instead of the whole protein. `--seed 0` fixes the random numbers, so everyone gets
the same answer. `EXHAUSTIVENESS` is how hard the program searches: 8 is the standard setting and
takes about three minutes for a molecule the size of silymarin. Lowering it to 4 halves the wait but
searches less thoroughly, and every compound in the notebook must use the same value for the scores
to be comparable.

In [ ]:
EXHAUSTIVENESS = 8

def dock(ligand_file, tag):
    """Dock one ligand into the receptor, inside the box around the reference ligand."""
    out_file = f"{WORK}/{tag}_docked.sdf"
    result = subprocess.run(
        [GNINA, "-r", f"{WORK}/receptor.pdb", "-l", ligand_file, "--autobox_ligand", f"{WORK}/ligand.sdf",
         "-o", out_file, "--seed", "0", "--exhaustiveness", str(EXHAUSTIVENESS)],
        capture_output=True, text=True, env=ENV)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-2000:])
    return out_file

dock(f"{WORK}/ligand.sdf", "silymarin")

gnina writes every pose it kept into one file, with the scores attached to each pose:

- **affinity**: the Vina score, an estimate of the binding energy in kcal/mol. More negative is better.
- **cnn_pose_score**: how confident the neural network is that the pose is correct, from 0 to 1.
- **cnn_affinity**: the neural network's estimate of the binding strength as a pK value. Higher is
  better, and 9 means nanomolar while 6 means micromolar.

The poses come out ranked by `cnn_pose_score`, which is gnina's own preference. Section 5 checks
whether that ranking is trustworthy here.

In [ ]:
import pandas as pd
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.warning")  # gnina's files are fine, RDKit just complains about them

def read_scores(docked_file):
    """Read the scores gnina attached to each pose in an output file."""
    rows = []
    for mol in Chem.SDMolSupplier(docked_file):
        if mol is not None:
            rows.append({"name": mol.GetProp("_Name") or "ligand",
                         "affinity": float(mol.GetProp("minimizedAffinity")),
                         "cnn_pose_score": float(mol.GetProp("CNNscore")),
                         "cnn_affinity": float(mol.GetProp("CNNaffinity"))})
    return pd.DataFrame(rows)

read_scores(f"{WORK}/silymarin_docked.sdf")

## 5. Check the redocked pose

The honest test is not the score but the position. We compare a redocked pose with the pose in the
file that was uploaded, using the root-mean-square deviation (RMSD), the average distance between
matching atoms in angstroms. Below 2 angstroms the two poses count as the same.

Two poses are worth comparing: the one the neural network ranks first, and the one with the best
Vina affinity. They are not always the same pose.

In [ ]:
from rdkit.Chem import rdMolAlign

def best_pose(docked_file, by="affinity"):
    """Return the best pose in a gnina output file, by Vina affinity or by CNN pose score."""
    poses = [mol for mol in Chem.SDMolSupplier(docked_file) if mol is not None]
    if by == "affinity":
        return min(poses, key=lambda mol: float(mol.GetProp("minimizedAffinity")))
    return max(poses, key=lambda mol: float(mol.GetProp("CNNscore")))

reference = Chem.MolFromMolFile(f"{WORK}/ligand.sdf")
for by in ("affinity", "cnn_pose_score"):
    rmsd = rdMolAlign.CalcRMS(best_pose(f"{WORK}/silymarin_docked.sdf", by), reference)
    print(f"best pose by {by:<15} RMSD {rmsd:5.2f} angstrom")
Chem.MolToMolFile(best_pose(f"{WORK}/silymarin_docked.sdf"), f"{WORK}/silymarin_best.sdf")

The pose with the best Vina affinity lands almost exactly on the uploaded one, so the search works:
gnina can find the right place in this pocket. The pose the neural network prefers is several
angstroms away, so its ranking cannot be trusted for this target. That is not surprising. The
network was trained on crystal structures, and CpABC1 here is a computed model. **From now on we
rank by the Vina affinity**, and keep the neural network scores as extra information.

> **Note:** this is why redocking comes first. Without it we would have ranked the whole library
> with a score that cannot even place silymarin correctly.

Now look at it. Drawing all 23,000 protein atoms would be slow, so we keep only the residues lining
the pocket and draw them as thin sticks. The uploaded pose is grey, the redocked pose is green. If
the two overlap, gnina found the pocket.

In [ ]:
import numpy as np

centre = reference.GetConformer().GetPositions().mean(axis=0)
coords = np.array([[float(line[30:38]), float(line[38:46]), float(line[46:54])] for line in protein])
pocket = [line for line, far in zip(protein, np.linalg.norm(coords - centre, axis=1) < 12) if far]
print(f"{len(pocket)} atoms within 12 angstrom of the ligand")

The viewer is interactive: drag to rotate, scroll to zoom.

In [ ]:
import py3Dmol

def view_pose(pose_file, reference_file=None):
    """Show a docked pose (green) in the pocket, next to a reference pose (grey)."""
    view = py3Dmol.view(width=700, height=450)
    view.addModel("\n".join(pocket), "pdb")
    view.setStyle({"stick": {"colorscheme": "whiteCarbon", "radius": 0.08}})
    if reference_file:
        view.addModel(open(reference_file).read(), "sdf")
        view.setStyle({"model": 1}, {"stick": {"colorscheme": "greyCarbon"}})
    view.addModel(open(pose_file).read(), "sdf")
    view.setStyle({"model": -1}, {"stick": {"colorscheme": "greenCarbon"}})
    view.zoomTo({"model": -1})
    return view

view_pose(f"{WORK}/silymarin_best.sdf", f"{WORK}/ligand.sdf")

## 6. Build the compound library

The library is a small set of public compounds that are chemically related to silymarin: the other
flavonolignans of the silymarin extract, plus simpler flavonoids that share the same core. They are
stored as SMILES strings in `data/silymarin_analogues.csv`, which is ordered so that the first ten
are a mix of large flavonolignans and small flavonoids. Docking takes a few minutes per compound, so
we screen those ten. Raise `LIBRARY_SIZE` to 16 to screen the whole file.

To screen your own compounds instead, set `UPLOAD_LIBRARY = True` and upload a CSV file with a
`name` column and a `smiles` column.

In [ ]:
LIBRARY_SIZE = 10
UPLOAD_LIBRARY = False

if UPLOAD_LIBRARY and "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    library = pd.read_csv(next(iter(uploaded)))
else:
    library = pd.read_csv("data/silymarin_analogues.csv").head(LIBRARY_SIZE)
library

A SMILES string says which atoms are bonded to which, but not where they are in space, and docking
needs a starting three-dimensional shape. RDKit builds one and relaxes it with a force field. The
random seed keeps the shapes the same every time the notebook runs.

> **Note:** we generate one shape per compound. Real screening campaigns generate several, because
> a flexible molecule can start from many shapes. gnina still explores the rotatable bonds itself.

In [ ]:
from rdkit.Chem import AllChem

def embed(smiles, name):
    """Turn a SMILES string into a single relaxed 3D structure."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp("_Name", name)
    return mol

molecules = [embed(s, n) for n, s in zip(library["name"], library["smiles"])]
print(f"{len(molecules)} molecules with 3D coordinates")

## 7. Dock the library

Every compound goes through exactly the same docking run as silymarin, into the same box. This is a
*virtual screen*: instead of testing compounds in the laboratory, we rank them on the computer first
and only take the best ones further.

Big flexible molecules take longer than small rigid ones, roughly one to three minutes each, so the
whole library takes about half an hour. The cell prints each result as it arrives, so you can watch
it go. Keep the browser tab open, otherwise Colab disconnects and the work is lost.

In [ ]:
import time

rows = []
for i, mol in enumerate(molecules, start=1):
    name = mol.GetProp("_Name")
    tag = name.replace(" ", "_")
    Chem.MolToMolFile(mol, f"{WORK}/{tag}.sdf")
    start = time.time()
    best = read_scores(dock(f"{WORK}/{tag}.sdf", tag)).sort_values("affinity").iloc[0]
    rows.append({"name": name, "affinity": best.affinity,
                 "cnn_pose_score": best.cnn_pose_score, "cnn_affinity": best.cnn_affinity})
    print(f"{i:2d}/{len(molecules)}  {name:<20} affinity {best.affinity:6.2f}  ({time.time() - start:.0f} s)")

results = pd.DataFrame(rows)

## 8. Rank the compounds

Section 5 showed that the Vina affinity is the score that places silymarin correctly in this
pocket, so that is what we rank on: the more negative, the better. Redocked silymarin gives us the
number to beat.

> **Note:** these are predictions, not measurements, and a docking score says nothing about whether
> a compound can be bought, whether it is safe, or whether it reaches the parasite. A compound that
> scores well here is worth testing, not a discovery.

In [ ]:
silymarin_affinity = read_scores(f"{WORK}/silymarin_docked.sdf")["affinity"].min()

ranked = results.sort_values("affinity").reset_index(drop=True)
ranked["beats_silymarin"] = ranked["affinity"] < silymarin_affinity
ranked.to_csv(f"{WORK}/docking_scores.csv", index=False)
print(f"silymarin: {silymarin_affinity:.2f} | compounds that beat it: {int(ranked['beats_silymarin'].sum())}")
ranked

The same numbers as a plot, with silymarin drawn as a horizontal line. The bars point downwards
because a binding energy is negative, so the compounds reaching furthest down are the best ones.

In [ ]:
import stylia

stylia.set_format("slide")
stylia.set_style("ersilia")
colors = stylia.NamedColors()

fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.bar(ranked["name"], ranked["affinity"],
       color=[colors.plum if b else colors.gray for b in ranked["beats_silymarin"]])
ax.axhline(silymarin_affinity, color=colors.blue)
ax.tick_params(axis="x", rotation=90)
stylia.label(ax, xlabel="", ylabel="Affinity (kcal/mol)")
stylia.save_figure(f"{WORK}/docking_scores.png")

Finally, look at the best compound in the pocket, with silymarin behind it in grey. Does it reach
the same parts of the pocket?

> **Exercise:** change `UPLOAD_LIBRARY` in section 6 to screen your own list of compounds, or set
> `EXHAUSTIVENESS` to 4 in section 4 and run the screen again. If a shorter search reshuffles the
> top of the list, the differences between those compounds were never real.

In [ ]:
top = ranked.loc[0, "name"]
top_pose = best_pose(f"{WORK}/{top.replace(' ', '_')}_docked.sdf")
Chem.MolToMolFile(top_pose, f"{WORK}/top_hit.sdf")
print(f"Best compound: {top} (affinity {ranked.loc[0, 'affinity']:.2f} kcal/mol)")
view_pose(f"{WORK}/top_hit.sdf", f"{WORK}/ligand.sdf")

## Summary

- We split the uploaded CpABC1-silymarin complex into a receptor and a reference ligand, and checked
  the setup by redocking silymarin into its own pocket.
- Redocking also told us which score to trust: the Vina affinity puts silymarin back where it
  belongs, the neural network score does not.
- We docked a library of silymarin analogues into the same pocket and ranked them by affinity, with
  redocked silymarin as the compound to beat.
- The ranking is saved in `work/docking_scores.csv`. Download it from the file browser on the left
  if you want to keep it, because the Colab session is wiped when it closes.

**Next:** take the compounds that beat silymarin and check whether they are drug-like and safe
enough to be worth making, and compare this ranking with the one from the pharmacophore model.